### Softmax从零开始实现

In [33]:
import torch
from IPython import display
from d2l import torch as d2l

# 加载数据迭代器
batch_size = 256
train_iter,test_iter = d2l.load_data_fashion_mnist(batch_size)

In [34]:
X,y = next(iter(train_iter))
X.shape[3]

28

In [35]:
num_inputs = X.shape[2]*X.shape[3] #28*28
num_outputs = 10

W = torch.normal(0,0.01,size=(num_inputs,num_outputs),requires_grad=True)
b = torch.zeros(num_outputs,requires_grad=True)

### Softmax

$ Softmax(X)_{ij} = \dfrac{e^{X_{ij}}}{\sum_k{e^{X_{ik}}}} $

In [36]:
def softmax(X):
    X_exp = torch.exp(X)
    partition = X_exp.sum(dim=1,keepdim=True)
    return X_exp / partition

In [37]:
X = torch.normal(0,1,(2,5))
X_prob = softmax(X)
X,X_prob,X_prob.sum(dim=1,keepdim=True)

(tensor([[-0.8647, -0.7897,  0.7406, -0.1064,  2.5145],
         [-0.2579,  1.0091, -0.6415, -0.5680, -0.3058]]),
 tensor([[0.0259, 0.0280, 0.1292, 0.0554, 0.7615],
         [0.1445, 0.5132, 0.0985, 0.1060, 0.1378]]),
 tensor([[1.],
         [1.]]))

### Softmax回归模型

In [38]:
def net(X):
    return softmax(torch.matmul(X.reshape(-1,W.shape[0]),W) + b)

### Loss Function交叉熵损失函数
$ -\sum\limits_{c=1}^C{y_clog(\hat{y}_c)} $

这里由于真实标签中y为真实的时候y就是1所以可以简化为

$ -\sum\limits_{c=1}^C{log{\hat{y}_c}} $ 

In [39]:
y = torch.tensor([0,2]) # 真实的标签
y_hat = torch.tensor([[0.1,0.3,0.6],[0.3,0.2,0.5]]) # 预测的每一个类别的概率
y_hat[[0,1],y] # 这里就是说我第一个真实的类是0然后我的预测解决是三个概率分别为0.1 0.3 0.6也就是说预测为真实标签0的概率为0.1,然后第二个的真实标签是2然后我预测第二类的概率是0.3,0.2,0.5也就说我的预测的标签为2的概率是0.2,本质上就是得到y_hat[0][0]和y_hat[2][2]

tensor([0.1000, 0.5000])

In [40]:
def cross_entropy(y_hat,y):
    return -torch.log(y_hat[range(len(y_hat)),y])
cross_entropy(y_hat,y)

tensor([2.3026, 0.6931])

### 将预测值类别与真实的y元素进行比较

In [41]:
def accuracy(y_hat,y):
    if len(y_hat.shape)>1 and y_hat.shape[1]>1:
        y_hat = y_hat.argmax(axis=1) # 将每一行中的最大值的下标存储到y_hat中
    cmp = y_hat.type(y.dtype) == y # 如果和真实的值相等的话cmp就为True
    return float(cmp.type(y.dtype).sum())

accuracy(y_hat,y)/len(y)

0.5

In [43]:
def evaluate_accuracy(net, data_iter):
    """计算在指定数据集上的模型的accuracy"""
    if isinstance(net,torch.nn.Module): # 如果这个net是一个模型的话就将这个net设置为评估模式
        net.eval()
    metric = Accumulator(2)
    with torch.no_grad():
        for X,y in data_iter:
            metric.add(accuracy(net(X),y),y.numel())
    return metric[0]/metric[1]

In [42]:
class Accumulator:
    """在n个变量上累加"""
    def __init__(self,n):
        self.data = [0.0]*n
    def add(self, *args):
        self.data = [a+float(b) for a,b in zip(self.data,args)]
    def reset(self):
        self.data = [0.0]*len(self.data)
    def __getitem__(self, key):
        return self.data[key]

In [45]:
evaluate_accuracy(net,test_iter)

0.0558